# VeriCharge — demo: reproduce the main results in a few minutes

This notebook reproduces the main tables and figures of the paper from the frozen artifacts, **without retraining anything**. It runs on a free CPU runtime in Google Colab or locally.

**What it shows**
1. The frozen VeriCharge (URS) selective policy on the 2020 FinalTest year (Table 3).
2. The benchmark against load-context baselines (Table 4, Figure 4).
3. Telemetry-fault behaviour and the validator (Table 7, Figures 6–7).

## 1. Setup
Set your GitHub user name below (only needed when running in Colab).

In [ ]:
GITHUB_USER = "Dr-Bashar-Uni-Hub"
REPO        = "VeriCharge"
RELEASE_TAG = "v1.0"

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=True)
    os.chdir(REPO)
print("Working directory:", os.getcwd())

In [ ]:
# Download and unpack the frozen artifacts (~35 MB) from the GitHub release
import zipfile, urllib.request
if not os.path.exists("artifacts/results"):
    url = f"https://github.com/{GITHUB_USER}/{REPO}/releases/download/{RELEASE_TAG}/vericharge_artifacts.zip"
    urllib.request.urlretrieve(url, "vericharge_artifacts.zip")
    zipfile.ZipFile("vericharge_artifacts.zip").extractall("artifacts")
os.environ["VERICHARGE_ARTIFACTS"] = os.path.abspath("artifacts")
os.environ["VERICHARGE_OUTPUTS"]   = os.path.abspath("outputs")
sys.path.insert(0, os.path.abspath("analysis"))
print(sorted(os.listdir("artifacts")))

## 2. Frozen VeriCharge policy on FinalTest (Table 3)
Thresholds were frozen on TrustCalibration (December 2019) and are applied unchanged to 2020.

In [ ]:
from common import *

rows = []
for j, h in enumerate(H):
    e = np.abs(test["y"][:, j] - test["yhat"][:, j])
    acc = test["T"][:, j] >= tau[j]
    fail = e > eps[j]
    rows.append({"Horizon": h, "Coverage": f"{acc.mean():.2%}",
                 "Full MAE (kW)": round(e.mean(), 3), "Accepted MAE (kW)": round(e[acc].mean(), 3),
                 "MAE reduction": f"{1 - e[acc].mean()/e.mean():.2%}",
                 "Full failure rate": f"{fail.mean():.2%}", "Accepted failure rate": f"{fail[acc].mean():.2%}"})
pd.DataFrame(rows)

## 3. Benchmark against load-context baselines (Table 4)
All learned gates are fitted on TrustFit and thresholded on TrustCalibration. The predicted-load rule needs no fitting.

In [ ]:
import runpy, warnings; warnings.filterwarnings("ignore")
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path("analysis/run_A1_baselines.py")
a1 = pd.read_csv("outputs/A1_baselines.csv")
show = ["URS (VeriCharge)", "U only (MC-dropout)", "Load context (LR)", "Predicted load ŷ (rule)", "Oracle", "Random / no selection"]
a1[a1.gater.isin(show)].pivot_table(index="gater", columns="horizon", values=["AUROC", "fail_at70"]).round(3)

## 4. Deep ensemble, telemetry faults and validator

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path("analysis/run_B1_ensemble.py")      # ~1 min on CPU
    runpy.run_path("analysis/run_B2_faults.py")
    runpy.run_path("analysis/run_validator.py")
v = pd.read_csv("outputs/V_validator_results.csv")
pol = ["URS (VeriCharge, frozen)", "Validator + URS (VeriCharge, frozen)"]
(v[v.policy.isin(pol)].pivot_table(index="scenario", columns=["policy", "horizon"], values="harmful_accept") * 100).round(1)

## 5. Figures

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path("analysis/make_figures.py")
from IPython.display import Image, display
for f in ["fig4_benchmark.png", "fig5_strata.png", "fig6_faults.png", "fig7_validator.png"]:
    display(Image(filename=f"outputs/figures/{f}", width=900))

The paired moving-block bootstrap (`analysis/run_A3_paired_bootstrap.py`, about 2–3 min) and the tolerance sensitivity (`analysis/run_A2_tolerance.py`) can be run the same way.